# Seven-day demand forecasting

Run the cells in order to train the weekly forecaster, evaluate it on all complete
held-out weeks, and regenerate the dashboard files. Run the data exploration notebook
first if the processed feature data is missing. The backtest uses recorded target-date
calendar and prices and only observes sales dated before each week's forecast origin.

In [ ]:
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / "src" / "forecast.py").is_file():
    PROJECT_DIR = PROJECT_DIR.parent
sys.path.insert(0, str(PROJECT_DIR))

from src.forecast import run_forecast, score

backtest, future_forecasts = run_forecast(PROJECT_DIR, write=True)
display(score(backtest).round(2).to_frame("Held-out performance"))

In [ ]:
display(backtest.groupby("horizon").agg(
    Model_MAE=("absolute_error", "mean"),
    Seasonal_Naive_MAE=("baseline_absolute_error", "mean"),
    Predictions=("sales", "count"),
).round(2))
display(backtest.groupby("item_id")["absolute_error"].mean().round(2).rename("MAE"))

In [ ]:
display(future_forecasts.pivot(
    index="date", columns="item_id", values="predicted_sales"
).round(2))